In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torch.optim as optim
import matplotlib.pyplot as plt

In [ ]:
df=pd.read_csv('/content/fmnist_small.csv')
df.sample(5)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
3132,6,0,0,0,0,0,0,0,0,0,...,106,2,0,1,0,0,0,0,0,0
5678,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1440,1,0,0,0,0,0,0,0,126,165,...,165,173,160,185,105,0,0,0,0,0
3441,3,0,0,0,0,0,0,0,0,0,...,144,0,0,2,0,0,0,0,0,0
4126,6,0,0,0,0,0,0,0,0,2,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
X = df.iloc[:,1:].values
y=df.iloc[:,0].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [ ]:
X_train = X_train/255.0
X_test = X_test/255.0

Create CustomDataset Class

In [ ]:
class customdataset(Dataset):
  def __init__(self,features,labels):
    self.features=torch.tensor(features, dtype=torch.float32)
    self.labels=torch.tensor(labels, dtype=torch.long)

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self,index):
    return self.features[index],self.labels[index]

In [ ]:
train_dataset = customdataset(X_train,y_train)

In [ ]:
test_dataset = customdataset(X_test, y_test)

Create dataset loader

In [ ]:
train_loader = DataLoader(train_dataset,batch_size=32, shuffle=True)
test_loader= DataLoader(test_dataset, batch_size= 32, shuffle=False)

Create Neural Network

In [ ]:
class ANN(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features,128),
        nn.ReLU(),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Linear(64,10)
    )
  def forward(self,x):
    return self.model(x)

In [ ]:
epochs = 100
learning_rate =0.1

In [ ]:
X_train.shape

(4800, 784)

In [ ]:
model = ANN(X_train.shape[1])
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(),lr =learning_rate)

In [ ]:
for epoch in range(epochs):
  total_epoch_loss = 0
  for batch_features, batch_labels in train_loader:
    outputs =model(batch_features)
    loss = criterion(outputs, batch_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_epoch_loss+=loss.item()
  avg_loss=total_epoch_loss/len(train_loader)
  print(f'Epoch:{epoch+1}, Loss:{avg_loss}')



Epoch:1, Loss:1.3381145838896433
Epoch:2, Loss:0.7751836534341177
Epoch:3, Loss:0.6616143316030503
Epoch:4, Loss:0.5916259607672691
Epoch:5, Loss:0.5467302531003952
Epoch:6, Loss:0.5112399723132451
Epoch:7, Loss:0.48459664583206175
Epoch:8, Loss:0.44904638638099037
Epoch:9, Loss:0.4285175785422325
Epoch:10, Loss:0.40471079319715497
Epoch:11, Loss:0.38360785990953444
Epoch:12, Loss:0.3689072991410891
Epoch:13, Loss:0.34822501997152966
Epoch:14, Loss:0.33014770835638046
Epoch:15, Loss:0.32444606239597
Epoch:16, Loss:0.3109476615488529
Epoch:17, Loss:0.301785063991944
Epoch:18, Loss:0.28627615615725516
Epoch:19, Loss:0.29640157279868923
Epoch:20, Loss:0.2723308929800987
Epoch:21, Loss:0.2677150381108125
Epoch:22, Loss:0.2506372179836035
Epoch:23, Loss:0.2442253198226293
Epoch:24, Loss:0.23280552906294663
Epoch:25, Loss:0.2295468017955621
Epoch:26, Loss:0.2181227067609628
Epoch:27, Loss:0.21166344627737999
Epoch:28, Loss:0.20373946066945792
Epoch:29, Loss:0.19001169194777806
Epoch:30, Loss

Evaluation

In [ ]:
model.eval()

ANN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [ ]:
total = 0
correct=0
with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    outputs=model(batch_features)
    _,predicted = torch.max(outputs, 1)
    total += batch_labels.shape[0]
    correct += (predicted==batch_labels).sum().item()
print(correct/total)

0.8308333333333333
